# Calibration targets and sparse time grids

Declarative `Target` / `TargetSet` objects turn scattered observations into a
`SavePlan` that only materialises those times. This chapter walks through
calendar-dated series (`Epoch` + `from_series`), fitting under `optax`, and
post-solve time tools — `at_times`, `resample`, and comparing sparse vs dense
footprints — that you use when the likelihood grid and the reporting grid
differ.

Likelihoods and priors are still WP10; here the contract stops at gather and
residuals.


In [ ]:
from datetime import date
from typing import Any, NamedTuple

import jax
import jax.numpy as jnp
import numpy as np
import optax
import pandas as pd

from summer4 import (
    Compartments,
    Epoch,
    FlowModel,
    Property,
    PropertyData,
    PropertyMap,
    SavePlan,
    SaveRequest,
    Target,
    TargetSet,
    TransitionFlow,
    derived_refs,
)


class Rates(NamedTuple):
    infection: float
    recovery: float


state = Property("state", ("S", "I", "R"))
pmap = PropertyMap.from_property(state)
refs = derived_refs(Rates)
model = FlowModel(pmap)
model.add_flow(TransitionFlow("infection", state["S"], state["I"], refs.infection))
model.add_flow(TransitionFlow("recovery", state["I"], state["R"], refs.recovery))
cm = model.compile()
y0 = PropertyData.wrap(pmap, np.array([999.0, 1.0, 0.0]))
true = Rates(infection=0.32, recovery=0.1)
epoch = Epoch(date(2020, 1, 1))
qty = Compartments(where=state["I"])


## Build targets from a dated pandas Series

`Target.from_series` maps a `DatetimeIndex` through the model `Epoch`. The
same object contributes its times into the save plan.


In [ ]:
# Truth on a fine grid, then sample ~36 irregular observation dates.
fine = SavePlan(requests={"I": SaveRequest(qty)})
truth_dense = cm.run(
    true, y0, t0=0.0, t1=120.0, dt=1.0, save=fine, solver="dopri5", epoch=epoch, rtol=1e-7, atol=1e-9
)
rng = np.random.default_rng(1)
obs_days = np.sort(rng.choice(np.arange(5, 115), size=36, replace=False)).astype(np.float64)
obs_vals = np.asarray(truth_dense["I"].at_times(obs_days).values.data).reshape(-1)
obs_dates = epoch.from_model(obs_days)
series = pd.Series(obs_vals, index=pd.DatetimeIndex(obs_dates))

target = Target.from_series("I", series, epoch, quantity=qty)
np.testing.assert_allclose(target.times, obs_days)
targets = TargetSet(targets=(target,))
sparse_plan = targets.plan(SavePlan())
assert sparse_plan.requests["I"].ts is not None
assert sparse_plan.requests["I"].ts.size == 36


## Memory: sparse likelihood grid vs dense reporting grid

`describe` takes the same `params` as `run`, so a FieldRef model does not
need a constant-rate twin just to size the plan.


In [ ]:
sparse_desc = cm.describe(sparse_plan, params=true, y0=y0, t0=0.0, dt=1.0, steps=120)
dense_desc = cm.describe(fine, params=true, y0=y0, t0=0.0, dt=1.0, steps=120)
print(f"sparse: {sparse_desc.total_nbytes:,} B")
print(f"dense:  {dense_desc.total_nbytes:,} B")
print(f"saved:  {dense_desc.total_nbytes - sparse_desc.total_nbytes:,} B")
assert sparse_desc.total_nbytes < dense_desc.total_nbytes


## Fit under the sparse plan

The loss closes over `targets.residuals`. Diffrax hits the observation times
exactly because `contribute` merged them into the request's `ts`.


In [ ]:
def loss(infection: Any) -> Any:
    params = Rates(infection=infection, recovery=true.recovery)
    res = cm.run(
        params,
        y0,
        t0=0.0,
        t1=120.0,
        dt=1.0,
        save=sparse_plan,
        solver="dopri5",
        epoch=epoch,
        rtol=1e-6,
        atol=1e-8,
    )
    return jnp.sum(jnp.asarray(targets.residuals(res)["I"]) ** 2)


opt = optax.adam(0.04)
infection = jnp.asarray(0.12)
opt_state = opt.init(infection)
value_and_grad = jax.jit(jax.value_and_grad(loss))

for _ in range(100):
    _value, grad = value_and_grad(infection)
    updates, opt_state = opt.update(grad, opt_state, infection)
    infection = optax.apply_updates(infection, updates)

recovered = float(infection)
print(f"true={true.infection:.3f}, recovered={recovered:.3f}")
assert abs(recovered - true.infection) / true.infection < 0.03


## Reporting run: dense save, then resample and `at_times`

A likelihood plan and a reporting plan are two `SavePlan`s — not two modes of
one shared model. After a dense solve, calendar `resample` and off-grid
`at_times` line the trajectory up with surveillance weeks or arbitrary dates.


In [ ]:
fitted = Rates(infection=recovered, recovery=true.recovery)
report = cm.run(
    fitted, y0, t0=0.0, t1=120.0, dt=1.0, save=fine, solver="dopri5", epoch=epoch
)
prevalence = report["I"]

# Weekly means via calendar resampling (needs an Epoch on the axis).
weekly = prevalence.resample("W", how="mean")
assert weekly.times.values.size < prevalence.times.values.size
assert weekly.times.values.size >= 12

# Sparse re-evaluation at the original observation dates — same path as gather.
at_obs = prevalence.at_times(target.times)
gathered = targets.gather(report)["I"]
np.testing.assert_allclose(
    np.asarray(at_obs.values.data).reshape(-1),
    np.asarray(gathered.values.data).reshape(-1),
    rtol=1e-5,
)

# Residuals against the fitted trajectory should be small at observation times.
resid = np.asarray(targets.residuals(report)["I"])
assert float(np.sqrt(np.mean(resid**2))) < 5.0
_ = prevalence.plot(legend=False)


## Two targets, one save group

Equal observation times collapse into a single `SubSaveAt` / save group, so I
and R at the same dates share one solve pass.


In [ ]:
from summer4.results.groups import group_requests

r_qty = Compartments(where=state["R"])
both_fine = SavePlan(requests={"I": SaveRequest(qty), "R": SaveRequest(r_qty)})
truth_both = cm.run(
    true, y0, t0=0.0, t1=120.0, dt=1.0, save=both_fine, solver="dopri5", epoch=epoch
)
r_obs = np.asarray(truth_both["R"].at_times(obs_days).values.data).reshape(-1)
r_target = Target.from_series(
    "R", pd.Series(r_obs, index=series.index), epoch, quantity=r_qty
)
pair = TargetSet(targets=(target, r_target))
pair_plan = pair.plan(SavePlan())
groups = group_requests(pair_plan, default_ts=np.array([0.0]))
assert len(groups) == 1
assert set(groups[0].keys) == {"I", "R"}
